### Momentum Feature

In [23]:
import pandas as pd
import sqlite3
from sector_rotation.config import SECTORS

# Open a connection to your database file
con = sqlite3.connect("../data/sector_rotation.db")

# Build the "?" placeholders for the SQL query — one per ticker
placeholders = ",".join("?" * len(SECTORS))

# The SQL query. In plain English:
#   "Give me the date, ticker, value columns FROM the prices table,
#    but ONLY rows where ticker is one of my 11,
#    sorted by date then ticker."
# The {placeholders} slots in that "?,?,?..." string via an f-string.
q = f"SELECT date, ticker, value FROM prices WHERE ticker IN ({placeholders}) ORDER BY date, ticker"

# Load into a df
long = pd.read_sql(q, con, params=SECTORS, parse_dates=["date"])

# Close the connection
con.close()

# ENSURE PROPER LOADING INTO DF
# print(long.shape)
# long["ticker"].value_counts()

# REORGANIZE THE DF SO THE DATA IS THE INDEX, ALL THE TICKERS ARE COLUMNS AND THE VALUES IN ARE PRICE VALUES
wide = long.pivot(index="date", columns="ticker", values="value")
wide = wide[SECTORS]

# ENSURE THE ABOVE CODE WORKS
# print(wide.shape)
# print(wide.isna().sum())
# wide.tail()
print(wide.index.dtype)

# RESAMPLE THE DATA SO THAT WE ONLY HAVE ONE PIECE OF DATA PER WEEK INSTEAD OF DAILY
weekly = wide.resample("W-FRI").last()

# ENSURE ABOVE CODE WORKS
# print(weekly.shape)
# print(weekly.index[-5:])
# weekly.tail()

mom_4 = weekly.pct_change(4, fill_method=None)
mom_12 = weekly.pct_change(12, fill_method=None)
mom_26 = weekly.pct_change(26, fill_method=None)

# CHECK MOM_4
# print(mom_4[["XLK", "XLRE", "XLC"]].tail())
# print(mom_4["XLC"].notna().idxmax())
# print(mom_4.loc["2018-07-06":"2018-07-27", ["XLC","XLK"]])

# CHECK MOM_12
print(mom_12[["XLK", "XLRE", "XLC"]].tail())
print(mom_12["XLC"].notna().idxmax())

# CHECK MOM_26
print(mom_26[["XLK", "XLRE", "XLC"]].tail())
print(mom_26["XLC"].notna().idxmax())



datetime64[us]
ticker           XLK      XLRE       XLC
date                                    
2026-06-19  0.473522  0.096226  0.022515
2026-06-26  0.333369  0.096653 -0.046894
2026-07-03  0.267734  0.052472 -0.035621
2026-07-10  0.205056  0.007978 -0.060147
2026-07-17  0.134661  0.025458 -0.016089
2018-09-14 00:00:00
ticker           XLK      XLRE       XLC
date                                    
2026-06-19  0.327546  0.106665 -0.054665
2026-06-26  0.238972  0.133668 -0.094692
2026-07-03  0.254507  0.123517 -0.056979
2026-07-10  0.274224  0.114421 -0.047655
2026-07-17  0.249950  0.071920 -0.009801
2018-12-21 00:00:00


### Write BAMLH0A0HYM2 to DB

In [4]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DB  = Path("../data/hy_oas_snapshot.db")
CSV = Path("../data/BAMLH0A0HYM2.csv")

# load the db version
con = sqlite3.connect(DB)
db = pd.read_sql("SELECT date, value FROM prices WHERE ticker='BAMLH0A0HYM2'", con)
con.close()

# load the csv version
csv = pd.read_csv(CSV)
csv.columns = ["date", "value"]
csv["value"] = pd.to_numeric(csv["value"], errors="coerce")  # "." holidays -> NaN
csv = csv.dropna(subset=["value"])

# force both date columns to identical string format
db["date"]  = pd.to_datetime(db["date"]).dt.strftime("%Y-%m-%d")
csv["date"] = pd.to_datetime(csv["date"]).dt.strftime("%Y-%m-%d")

# outer merge: see overlap AND rows unique to each side
merged = db.merge(csv, on="date", how="outer", suffixes=("_db", "_csv"), indicator=True)
print(merged["_merge"].value_counts())

# compare values only where both sources have the date
both = merged[merged["_merge"] == "both"].copy()
both["match"] = np.isclose(both["value_db"], both["value_csv"], atol=1e-6)
print("overlap rows:", len(both))
print("mismatches:", (~both["match"]).sum())

both[~both["match"]]   # shows the disagreeing rows, empty if all match

_merge
both          6322
right_only    1208
left_only        0
Name: count, dtype: int64
overlap rows: 6322
mismatches: 0


,date,value_db,value_csv,_merge,match


### Testing

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

MAIN_DB = Path("../data/sector_rotation.db")

con = sqlite3.connect(MAIN_DB)
df = pd.read_sql(
    "SELECT date, value FROM prices WHERE ticker='BAMLH0A0HYM2' ORDER BY date",
    con,
)
con.close()

df["date"] = pd.to_datetime(df["date"])

# --- basic coverage ---
print("rows:      ", len(df))
print("min date:  ", df["date"].min().date())
print("max date:  ", df["date"].max().date())
print("duplicates:", df["date"].duplicated().sum())

# --- gap analysis ---
# gap = calendar days between consecutive rows.
# 1 = next day, 3 = Fri->Mon (normal weekend), 4 = Mon holiday, etc.
df["gap"] = df["date"].diff().dt.days

# anything > 4 days is bigger than a weekend + one holiday — worth a look
big = df[df["gap"] > 4].copy()
print("\ngaps larger than 4 days:", len(big))
big["prev_date"] = df["date"].shift(1)
print(big[["prev_date", "date", "gap"]].to_string(index=False))